# NVIDIA RAPIDS vs CPU — Benchmark Comparison
### Part 4

[RAPIDS](https://rapids.ai/) runs data science on the GPU: **cuDF** mirrors the pandas API and
**cuML** mirrors the scikit-learn API, so the same code runs — just far faster on large data.
This notebook benchmarks GPU (RAPIDS) against CPU (pandas / scikit-learn) on identical tasks.

**Benchmarks:**
1. DataFrame ops: cuDF vs pandas
2. K-Means: cuML vs scikit-learn
3. Random Forest: cuML vs scikit-learn
4. Linear/Logistic Regression: cuML vs scikit-learn
5. PCA: cuML vs scikit-learn
6. Speedup summary chart

> ⚠️ **GPU REQUIRED.** In Colab: `Runtime → Change runtime type → T4 GPU`, then Run all.
> RAPIDS will NOT run on a CPU-only runtime.

In [ ]:
# Confirm a GPU is attached
!nvidia-smi

In [ ]:
# Install RAPIDS (cuDF + cuML) matching Colab's CUDA 12 runtime.
!pip install -q cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com
print("RAPIDS installed. If import fails, restart runtime and Run all again.")

In [ ]:
import time, numpy as np, pandas as pd
import matplotlib.pyplot as plt

import cudf, cuml
import cupy as cp
print("cuDF:", cudf.__version__, "| cuML:", cuml.__version__)

def timeit(fn, *a, **k):
    t0 = time.time(); out = fn(*a, **k)
    # sync GPU work before stopping the clock
    try: cp.cuda.Stream.null.synchronize()
    except Exception: pass
    return out, time.time() - t0

results = {}  # task -> (cpu_time, gpu_time)

## 1. DataFrame operations — cuDF vs pandas

A groupby-aggregate over several million rows.

In [ ]:
N = 10_000_000
rng = np.random.default_rng(0)
data = {"key": rng.integers(0, 100, N), "val": rng.random(N)}

pdf = pd.DataFrame(data)
_, t_cpu = timeit(lambda: pdf.groupby("key")["val"].mean())

gdf = cudf.DataFrame(data)
_, t_gpu = timeit(lambda: gdf.groupby("key")["val"].mean())

results["DataFrame groupby"] = (t_cpu, t_gpu)
print(f"pandas: {t_cpu:.3f}s  |  cuDF: {t_gpu:.3f}s  |  speedup: {t_cpu/t_gpu:.1f}x")

## 2. K-Means — cuML vs scikit-learn

In [ ]:
from sklearn.cluster import KMeans as skKMeans
from cuml.cluster import KMeans as cuKMeans
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=1_000_000, centers=8, n_features=20, random_state=0)
X = X.astype(np.float32)

_, t_cpu = timeit(lambda: skKMeans(n_clusters=8, n_init=3, max_iter=100,
                                   random_state=0).fit(X))
X_gpu = cp.asarray(X)
_, t_gpu = timeit(lambda: cuKMeans(n_clusters=8, max_iter=100,
                                   random_state=0).fit(X_gpu))

results["K-Means"] = (t_cpu, t_gpu)
print(f"sklearn: {t_cpu:.3f}s  |  cuML: {t_gpu:.3f}s  |  speedup: {t_cpu/t_gpu:.1f}x")

## 3. Random Forest — cuML vs scikit-learn

In [ ]:
from sklearn.ensemble import RandomForestClassifier as skRF
from cuml.ensemble import RandomForestClassifier as cuRF
from sklearn.datasets import make_classification

Xc, yc = make_classification(n_samples=200_000, n_features=50,
                             n_informative=20, n_classes=2, random_state=0)
Xc = Xc.astype(np.float32); yc = yc.astype(np.int32)

_, t_cpu = timeit(lambda: skRF(n_estimators=100, max_depth=16,
                               n_jobs=-1, random_state=0).fit(Xc, yc))
Xc_gpu, yc_gpu = cp.asarray(Xc), cp.asarray(yc)
_, t_gpu = timeit(lambda: cuRF(n_estimators=100, max_depth=16).fit(Xc_gpu, yc_gpu))

results["Random Forest"] = (t_cpu, t_gpu)
print(f"sklearn: {t_cpu:.3f}s  |  cuML: {t_gpu:.3f}s  |  speedup: {t_cpu/t_gpu:.1f}x")

## 4. Logistic Regression — cuML vs scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression as skLR
from cuml.linear_model import LogisticRegression as cuLR

_, t_cpu = timeit(lambda: skLR(max_iter=1000).fit(Xc, yc))
_, t_gpu = timeit(lambda: cuLR(max_iter=1000).fit(Xc_gpu, yc_gpu))

results["Logistic Regression"] = (t_cpu, t_gpu)
print(f"sklearn: {t_cpu:.3f}s  |  cuML: {t_gpu:.3f}s  |  speedup: {t_cpu/t_gpu:.1f}x")

## 5. PCA — cuML vs scikit-learn

In [ ]:
from sklearn.decomposition import PCA as skPCA
from cuml.decomposition import PCA as cuPCA

Xp = np.random.RandomState(0).rand(500_000, 100).astype(np.float32)
_, t_cpu = timeit(lambda: skPCA(n_components=10).fit_transform(Xp))
Xp_gpu = cp.asarray(Xp)
_, t_gpu = timeit(lambda: cuPCA(n_components=10).fit_transform(Xp_gpu))

results["PCA"] = (t_cpu, t_gpu)
print(f"sklearn: {t_cpu:.3f}s  |  cuML: {t_gpu:.3f}s  |  speedup: {t_cpu/t_gpu:.1f}x")

## 6. Speedup summary

Side-by-side times and the GPU speedup factor per task.

In [ ]:
tasks = list(results.keys())
cpu_t = [results[t][0] for t in tasks]
gpu_t = [results[t][1] for t in tasks]
speedups = [c / g for c, g in zip(cpu_t, gpu_t)]

summary = pd.DataFrame({"CPU (s)": cpu_t, "GPU (s)": gpu_t,
                        "Speedup (x)": speedups}, index=tasks).round(2)
print(summary)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(tasks)); w = 0.35
ax[0].bar(x - w/2, cpu_t, w, label="CPU"); ax[0].bar(x + w/2, gpu_t, w, label="GPU")
ax[0].set_xticks(x); ax[0].set_xticklabels(tasks, rotation=30, ha="right")
ax[0].set_ylabel("seconds (log)"); ax[0].set_yscale("log")
ax[0].set_title("Runtime: CPU vs GPU"); ax[0].legend()

ax[1].bar(tasks, speedups, color="green")
ax[1].set_xticklabels(tasks, rotation=30, ha="right")
ax[1].set_ylabel("x faster"); ax[1].set_title("GPU speedup factor")
for i, s in enumerate(speedups):
    ax[1].text(i, s, f"{s:.0f}x", ha="center", va="bottom")
plt.tight_layout(); plt.show()

## Summary

RAPIDS delivers large speedups by moving the pandas/scikit-learn workflow onto the GPU **with
essentially the same API** — the imports change (`cudf`/`cuml` instead of `pandas`/`sklearn`), the
code barely does. Speedups grow with data size; on small data the CPU can win due to GPU transfer
overhead, so RAPIDS pays off most on large datasets.